In [1]:
# ==========================================
# INSTALLATION (RUN THIS FIRST)
# ==========================================
!pip install -q transformers==4.36.0
!pip install -q sentencepiece
!pip install -q torch

print("✅ Packages installed")

# ==========================================
# CAFA Akkadian Translation - FIXED (No Tensor Errors)
# ⭐ FIXED: Long input truncation, regex patterns, error handling
# ==========================================

import os
import re
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
    AutoModel, LogitsProcessor
)
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("🏺 AKKADIAN TRANSLATION - FIXED (TENSOR-SAFE)")
print("="*70)

# ==========================================
# CONFIGURATION
# ==========================================
class CFG:
    # Data
    DATA_PATH = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
    OUTPUT_PATH = "submission.csv"
    
    # Models - Original 3 ByT5 checkpoints
    BYT5_MODELS = [
        "/kaggle/input/byt5-base-big-data2",
        "/kaggle/input/byt5-akkadian-model",
        "/kaggle/input/train-gap-all-2/byt5-base-akkadian_gap_setence2"
    ]
    
    # Local Kaggle model paths
    QWEN_PATH = "/kaggle/input/qwen2-5-7b"
    
    # Akkadian dictionary (expanded)
    LEXICON = {
        "šarru": ["king"], "šarratu": ["queen"],
        "māru": ["son"], "mārtu": ["daughter"],
        "bītu": ["house", "temple", "household"],
        "kaspum": ["silver", "money"], 
        "ālum": ["city", "town"],
        "awīlum": ["man", "person"],
        "ilum": ["god"], "ištaru": ["goddess"],
        "šamû": ["heaven", "sky"], "erṣetu": ["earth"],
        "wardum": ["slave", "servant"], "amtum": ["female slave"],
        "tuppum": ["tablet"], "ṭuppum": ["tablet"],
        "kānišum": ["Kanesh"], "aššur": ["Ashur"],
        "šamšum": ["sun"], "šīnum": ["silver"],
    }
    
    # ⭐ SAFETY LIMITS
    MAX_INPUT_LENGTH = 800  # Chars before truncation
    MAX_OUTPUT_LENGTH = 512  # Tokens
    
    # Optimization flags
    USE_ENSEMBLE = True
    USE_CONSTRAINED_DECODING = True
    USE_LLM_POSTEDIT = True
    
    # Hardware
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    BATCH_SIZE = 4 if torch.cuda.is_available() else 1
    NUM_WORKERS = 2
    
    # Generation parameters
    NUM_BEAMS = 12
    NUM_RETURN_SEQUENCES = 5
    MAX_NEW_TOKENS = 512
    LENGTH_PENALTY = 1.10
    
    # Template markers for cleaning
    TEMPLATE_MARKERS = [
        "You are an expert",
        "RULES:",
        "Given the following",
        "Source:",
        "Translate it",
        "Do not add explanations",
        "Be concise and clear"
    ]

print(f"Device: {CFG.DEVICE}")
print(f"Optimizations enabled:")
print(f"  - Ensemble: {CFG.USE_ENSEMBLE}")
print(f"  - Constrained Decoding: {CFG.USE_CONSTRAINED_DECODING}")
print(f"  - LLM Post-Edit: {CFG.USE_LLM_POSTEDIT}")
print(f"  - ⭐ Input Safety: ENABLED (max {CFG.MAX_INPUT_LENGTH} chars)")
print("="*70)

# ==========================================
# PREPROCESSING (⭐ FIXED)
# ==========================================
def preprocess_transliteration(text):
    """Preprocess with safety checks for extreme inputs"""
    if pd.isna(text): 
        return ""
    
    processed = str(text)
    
    # ⭐ FIX 1: TRUNCATE EXTREME INPUTS
    if len(processed) > CFG.MAX_INPUT_LENGTH:
        print(f"  ⚠️ Truncating long input: {len(processed)} → {CFG.MAX_INPUT_LENGTH} chars")
        processed = processed[:CFG.MAX_INPUT_LENGTH] + " <big_gap>"
    
    # ⭐ FIX 2: MORE SPECIFIC GAP PATTERNS (avoid "bigggg" in names)
    # Only match ACTUAL gap notation, not repeated letters in names
    processed = re.sub(r'(\[\.{3,}\]|…{2,}|\.{5,})', '<big_gap>', processed)
    processed = re.sub(r'(xx+|\s+x\s+|\[x+\])', '<gap>', processed)
    
    # Clean up consecutive gaps
    processed = re.sub(r'<gap>(\s*<gap>)+', '<gap>', processed)
    processed = re.sub(r'<big_gap>(\s*<big_gap>)+', '<big_gap>', processed)
    
    return processed.strip()

def extract_akkadian_words(text):
    """Extract potential Akkadian words for dictionary lookup"""
    clean = re.sub(r'<\w+>', '', text)
    words = re.findall(r'\b[a-zšṣṭḫāēīū]+\b', clean, re.IGNORECASE)
    return words

# ==========================================
# POSTPROCESSING (Enhanced)
# ==========================================
def postprocess_translation(text):
    if not isinstance(text, str) or not text.strip(): 
        return ""
    
    processed = text.replace('ḫ', 'h').replace('Ḫ', 'H')
    sub_map = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    processed = processed.translate(sub_map)

    processed = re.sub(r'(\[x\]|\(x\)|\bx\b)', '<gap>', processed, flags=re.I)
    processed = re.sub(r'(\.{3,}|…|\[\.+\])', '<big_gap>', processed)
    
    processed = re.sub(r'<gap>\s*<gap>', ' <big_gap> ', processed)
    processed = re.sub(r'<big_gap>\s*<big_gap>', ' <big_gap> ', processed)

    processed = re.sub(r'\((fem|plur|pl|sing|singular|plural|\?|!)\.?\s*\w*\)', '', processed, flags=re.I)

    processed = processed.replace('<gap>', '\x00GAP\x00').replace('<big_gap>', '\x00BIG\x00')
    
    bad_chars = '!?()"—–<>⌈⌋⌊[]+ʾ/;'
    processed = processed.translate(str.maketrans('', '', bad_chars))

    processed = processed.replace('\x00GAP\x00', ' <gap> ').replace('\x00BIG\x00', ' <big_gap> ')

    frac_map = {
        r'\.5\b': ' ½', r'\.25\b': ' ¼', r'\.75\b': ' ¾',
        r'\.33+\d*\b': ' ⅓', r'\.66+\d*\b': ' ⅔'
    }
    for pat, rep in frac_map.items():
        processed = re.sub(r'(\d+)' + pat, r'\1' + rep, processed)
        processed = re.sub(r'\b0' + pat, rep.strip(), processed)

    # Remove repeated words/phrases
    processed = re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', processed)
    for n in range(4, 1, -1):
        pat = r'\b((?:\w+\s+){' + str(n-1) + r'}\w+)(?:\s+\1\b)+'
        processed = re.sub(pat, r'\1', processed)

    return re.sub(r'\s+', ' ', processed).strip().strip('-')

# ⭐ Clean template leakage
def remove_template_contamination(text, source_text):
    """Remove system prompt and source code if they leak into output"""
    original_text = text
    
    # Remove template markers
    for marker in CFG.TEMPLATE_MARKERS:
        if marker in text:
            parts = text.split(marker)
            text = parts[0].strip()
            print(f"      ⚠️ Removed template marker: '{marker[:30]}...'")
    
    # Remove source text if it leaked
    akkadian_pattern = r'\b[a-zšṣṭḫāēīū]{2,}-[a-zšṣṭḫāēīū]{2,}'
    if re.search(akkadian_pattern, text):
        sentences = text.split('.')
        clean_sentences = []
        for sent in sentences:
            if re.search(akkadian_pattern, sent):
                break
            clean_sentences.append(sent)
        
        if clean_sentences:
            text = '. '.join(clean_sentences)
            if not text.endswith('.'):
                text += '.'
            print(f"      ⚠️ Removed source text leakage")
    
    # Safety check: if we removed too much, return original
    if len(text) < 10 and len(original_text) > 50:
        print(f"      ⚠️ Cleaning removed too much, using ByT5 output")
        return None
    
    return text

# ==========================================
# DATASET
# ==========================================
class AkkadianDataset(Dataset):
    def __init__(self, dataframe):
        self.ids = dataframe['id'].tolist()
        self.transliterations = dataframe['transliteration'].tolist()
        self.texts = ["translate Akkadian to English: " + str(t) for t in self.transliterations]
    
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, idx):
        return self.ids[idx], self.texts[idx], self.transliterations[idx]

# ==========================================
# CONSTRAINED DECODING
# ==========================================
class LexiconConstraintProcessor(LogitsProcessor):
    """Boost probabilities of dictionary translations"""
    def __init__(self, tokenizer, lexicon, source_words):
        self.tokenizer = tokenizer
        self.lexicon = lexicon
        self.source_words = source_words
        self.boost_tokens = self._get_boost_tokens()
    
    def _get_boost_tokens(self):
        boost = set()
        for word in self.source_words:
            if word.lower() in self.lexicon:
                for translation in self.lexicon[word.lower()]:
                    token_ids = self.tokenizer.encode(translation, add_special_tokens=False)
                    boost.update(token_ids)
        return boost
    
    def __call__(self, input_ids, scores):
        if len(self.boost_tokens) > 0:
            for token_id in self.boost_tokens:
                if token_id < scores.shape[1]:
                    scores[:, token_id] += 2.0
        return scores

# ==========================================
# MODEL LOADING
# ==========================================
def load_models():
    """Load all models from local Kaggle paths"""
    print("\n[1/6] Loading models...")
    
    models = []
    tokenizer = None
    
    for i, path in enumerate(CFG.BYT5_MODELS):
        print(f"  Loading ByT5 model {i+1}/3: {path.split('/')[-1]}...")
        model = AutoModelForSeq2SeqLM.from_pretrained(path)
        model = model.to(CFG.DEVICE).eval()
        models.append(model)
        
        if tokenizer is None:
            tokenizer = AutoTokenizer.from_pretrained(path)
    
    print(f"  ✅ ByT5 models loaded")
    
    llm_model = None
    llm_tokenizer = None
    if CFG.USE_LLM_POSTEDIT:
        try:
            print(f"  Loading Qwen from {CFG.QWEN_PATH}...")
            llm_tokenizer = AutoTokenizer.from_pretrained(CFG.QWEN_PATH, trust_remote_code=True)
            llm_model = AutoModelForCausalLM.from_pretrained(
                CFG.QWEN_PATH,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None,
                trust_remote_code=True
            )
            llm_model.eval()
            print(f"  ✅ Qwen loaded")
        except Exception as e:
            print(f"  ⚠️  Qwen loading failed: {e}")
            CFG.USE_LLM_POSTEDIT = False
    
    print("  ✅ All models loaded successfully")
    return models, tokenizer, llm_model, llm_tokenizer

# ==========================================
# ENSEMBLE GENERATION (⭐ WITH ERROR HANDLING)
# ==========================================
@torch.inference_mode()
def generate_ensemble_candidates(models, tokenizer, inputs, source_words):
    """Generate n-best candidates from all models"""
    candidates = []
    
    for model_idx, model in enumerate(models):
        try:
            logits_processors = []
            if CFG.USE_CONSTRAINED_DECODING:
                constraint_proc = LexiconConstraintProcessor(tokenizer, CFG.LEXICON, source_words)
                logits_processors.append(constraint_proc)
            
            outputs = model.generate(
                input_ids=inputs.input_ids.to(CFG.DEVICE),
                attention_mask=inputs.attention_mask.to(CFG.DEVICE),
                num_beams=CFG.NUM_BEAMS,
                num_return_sequences=CFG.NUM_RETURN_SEQUENCES,
                max_new_tokens=CFG.MAX_NEW_TOKENS,
                length_penalty=CFG.LENGTH_PENALTY,
                output_scores=True,
                return_dict_in_generate=True,
                logits_processor=logits_processors if logits_processors else None
            )
            
            for seq, score in zip(outputs.sequences, outputs.sequences_scores):
                decoded = tokenizer.decode(seq, skip_special_tokens=True)
                candidates.append({
                    'text': decoded,
                    'model_score': score.item(),
                    'model_id': model_idx
                })
        except Exception as e:
            print(f"      ⚠️ Model {model_idx} failed: {e}")
            continue
    
    # Return best by score, or fallback
    if candidates:
        return max(candidates, key=lambda x: x['model_score'])
    else:
        return {'text': '<gap> translation failed <gap>', 'model_score': 0.0, 'model_id': -1}

# ==========================================
# LLM POST-EDITING
# ==========================================
@torch.inference_mode()
def llm_post_edit(translation, source, llm_model, llm_tokenizer):
    """Refine translation with Qwen - NO TEMPLATE CONTAMINATION"""
    if not CFG.USE_LLM_POSTEDIT or llm_model is None:
        return translation
    
    try:
        # Ultra-short prompt to prevent leakage
        system_msg = "You are an Akkadian translator. Fix grammar errors in the English translation. Keep <gap> tokens. Output only the corrected translation, nothing else."
        
        user_msg = f"Translation: {translation}\n\nCorrected:"
        
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ]
        
        text = llm_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        inputs = llm_tokenizer([text], return_tensors="pt", max_length=768, truncation=True)
        inputs = {k: v.to(llm_model.device) for k, v in inputs.items()}
        
        # Conservative generation
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=llm_tokenizer.pad_token_id or llm_tokenizer.eos_token_id,
            eos_token_id=llm_tokenizer.eos_token_id
        )
        
        # Only decode new tokens
        input_length = inputs['input_ids'].shape[1]
        generated_ids = outputs[0][input_length:]
        response = llm_tokenizer.decode(generated_ids, skip_special_tokens=True)
        corrected = response.strip()
        
        # Aggressive template removal
        corrected = remove_template_contamination(corrected, source)
        
        if corrected is None:
            return translation
        
        # Clean common artifacts
        if "Corrected:" in corrected:
            corrected = corrected.split("Corrected:")[-1].strip()
        if "Translation:" in corrected:
            corrected = corrected.split("Translation:")[-1].strip()
        
        # Remove incomplete sentences at the end
        if corrected and not corrected[-1] in '.!?':
            last_period = max(corrected.rfind('.'), corrected.rfind('!'), corrected.rfind('?'))
            if last_period > len(corrected) * 0.5:
                corrected = corrected[:last_period + 1]
        
        # Safety checks
        if len(corrected) < 5:
            print(f"      ⚠️ LLM output too short, using original")
            return translation
        
        if len(corrected) > len(translation) * 2.5:
            print(f"      ⚠️ LLM output too long, using original")
            return translation
        
        # Check if gaps were preserved
        source_has_gaps = '<gap>' in translation or '<big_gap>' in translation
        output_has_gaps = '<gap>' in corrected or '<big_gap>' in corrected
        
        if source_has_gaps and not output_has_gaps:
            print(f"      ⚠️ LLM removed gaps, using original")
            return translation
        
        return corrected
    
    except Exception as e:
        print(f"    ⚠️  LLM post-edit failed: {e}")
        return translation

# ==========================================
# MAIN PIPELINE (⭐ WITH FULL ERROR HANDLING)
# ==========================================
def main():
    import time
    start_time = time.time()
    
    print("\n[2/6] Loading data...")
    df = pd.read_csv(CFG.DATA_PATH)
    df['transliteration'] = df['transliteration'].apply(preprocess_transliteration)
    print(f"  Loaded {len(df)} samples")
    
    models, tokenizer, llm_model, llm_tokenizer = load_models()
    
    dataset = AkkadianDataset(df)
    dataloader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0
    )
    
    print("\n[3/6] Running translation pipeline...")
    results = []
    failed_count = 0
    
    for sample_id, text, source in tqdm(dataloader, desc="  Translating"):
        # ✅ FIX: Convert tensors to Python types
        sample_id = int(sample_id[0])  # tensor → int
        text = text[0]  # List → string
        source = source[0]  # List → string
        
        # ⭐ FIX 3: FULL TRY-CATCH PER SAMPLE
        try:
            source_words = extract_akkadian_words(source)
            inputs = tokenizer(text, return_tensors="pt", max_length=CFG.MAX_OUTPUT_LENGTH, truncation=True)
            
            # Safety check: ensure inputs are valid
            if inputs.input_ids.shape[1] == 0:
                raise ValueError("Empty input after tokenization")
            
            # Strategy 1: Ensemble
            if CFG.USE_ENSEMBLE:
                best_candidate = generate_ensemble_candidates(models, tokenizer, inputs, source_words)
                translation = best_candidate['text']
            else:
                outputs = models[0].generate(
                    input_ids=inputs.input_ids.to(CFG.DEVICE),
                    attention_mask=inputs.attention_mask.to(CFG.DEVICE),
                    num_beams=CFG.NUM_BEAMS,
                    max_new_tokens=CFG.MAX_NEW_TOKENS,
                    length_penalty=CFG.LENGTH_PENALTY
                )
                translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            translation = postprocess_translation(translation)
            
            # Strategy 2: LLM refinement
            if CFG.USE_LLM_POSTEDIT and len(translation) > 10:
                refined = llm_post_edit(translation, source, llm_model, llm_tokenizer)
                refined = postprocess_translation(refined)
                
                # Use refined only if it's reasonable
                if 10 < len(refined) < len(translation) * 2:
                    translation = refined
            
            # Final safety: ensure non-empty
            if not translation or len(translation) < 3:
                translation = "<gap> translation unavailable <gap>"
            
            # ✅ Append as tuple of (int, str)
            results.append((sample_id, translation))
        
        except Exception as e:
            failed_count += 1
            print(f"\n    ❌ Failed on {sample_id}: {e}")
            
            # ⭐ SMART FALLBACK
            fallback_parts = []
            
            if source_words:
                known_translations = []
                for word in source_words[:5]:
                    if word.lower() in CFG.LEXICON:
                        known_translations.extend(CFG.LEXICON[word.lower()])
                
                if known_translations:
                    fallback_parts.append(' '.join(known_translations[:3]))
            
            numbers = re.findall(r'\d+', source)
            if numbers:
                fallback_parts.append(f"involving {numbers[0]}")
            
            if fallback_parts:
                fallback = ' '.join(fallback_parts) + " <big_gap>"
            else:
                fallback = "From <gap> to <gap> <big_gap>"
            
            # ✅ Append as tuple of (int, str)
            results.append((sample_id, fallback))
            continue
    
    print(f"\n  Translation complete: {len(results)} total, {failed_count} failures")
    
    print("\n[4/6] Creating submission...")
    submission = pd.DataFrame(results, columns=['id', 'translation'])
    
    print("\n[5/6] Quality checks...")
    print(f"  Total translations: {len(submission)}")
    print(f"  Avg length: {submission['translation'].str.len().mean():.1f} chars")
    print(f"  Empty translations: {(submission['translation'].str.len() == 0).sum()}")
    print(f"  Has <gap> tokens: {submission['translation'].str.contains('<gap>').sum()}")
    
    print("\n[6/6] Saving submission...")
    submission.to_csv(CFG.OUTPUT_PATH, index=False)
    
    total_time = (time.time() - start_time) / 60
    
    print("\n✅ Complete!")
    print("="*70)
    print(f"📁 Output: {CFG.OUTPUT_PATH}")
    print(f"⏱️  Total Time: {total_time:.1f} minutes")
    print(f"❌ Failed: {failed_count}/{len(submission)}")
    print(f"\n📊 Sample translations:")
    print(submission.head(10).to_string(index=False))
    print("="*70)
    
    return submission

if __name__ == '__main__':
    result = main()


ERROR: Could not find a version that satisfies the requirement transformers==4.36.0 (from versions: none)
ERROR: No matching distribution found for transformers==4.36.0
✅ Packages installed
🏺 AKKADIAN TRANSLATION - FIXED (TENSOR-SAFE)
Device: cuda
Optimizations enabled:
  - Ensemble: True
  - Constrained Decoding: True
  - LLM Post-Edit: True
  - ⭐ Input Safety: ENABLED (max 800 chars)

[2/6] Loading data...
  Loaded 4 samples

[1/6] Loading models...
  Loading ByT5 model 1/3: byt5-base-big-data2...


2026-01-24 15:17:00.731615: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769267820.949231      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769267821.014193      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769267821.538907      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769267821.538958      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769267821.538961      24 computation_placer.cc:177] computation placer alr

  Loading ByT5 model 2/3: byt5-akkadian-model...
  Loading ByT5 model 3/3: byt5-base-akkadian_gap_setence2...
  ✅ ByT5 models loaded
  Loading Qwen from /kaggle/input/qwen2-5-7b...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  ✅ Qwen loaded
  ✅ All models loaded successfully

[3/6] Running translation pipeline...


  Translating:   0%|          | 0/4 [00:00<?, ?it/s]

      ⚠️ LLM output too long, using original
      ⚠️ LLM output too long, using original

  Translation complete: 4 total, 0 failures

[4/6] Creating submission...

[5/6] Quality checks...
  Total translations: 4
  Avg length: 185.8 chars
  Empty translations: 0
  Has <gap> tokens: 0

[6/6] Saving submission...

✅ Complete!
📁 Output: submission.csv
⏱️  Total Time: 3.7 minutes
❌ Failed: 0/4

📊 Sample translations:
 id                                                                                                                                                                                                                                                                                                     translation
  0                                                                                                                                                               From the Kanesh colony to the entrusted traders and our messengers, every single day and every shekel of silver: